### Phase 5: Evaluation & Vergleich (Der Kern der Thesis)
Beweis des Mehrwerts zur Reduktion des **Sequence of Returns Risk**.
*   **Metriken:**
    *   **Maximum Drawdown (MDD):** Wie tief war der maximale Fall? (Wichtigstes Ziel).
    *   **Sharpe Ratio / Sortino Ratio:** Risiko-adjustierte Rendite.
    *   **Calmar Ratio:** Rendite im Verhältnis zum Drawdown.
    *   **Regime-Stabilität:** Wie oft schaltet das Modell um? (LSTMs neigen zum "Overfitting" und Rauschen).

In [1]:
import pandas as pd

# Daten aus dem data-Ordner laden
backtesting_results = pd.read_parquet("../data/04_backtesting_results_data.parquet")

In [2]:
import pandas as pd
import numpy as np

# Daten aus dem data-Ordner laden
test_df = pd.read_parquet("../data/03_test_df_data.parquet")
backtesting_results = pd.read_parquet("../data/04_backtesting_results_data.parquet")

def evaluate_strategies(results_df, trades_df):
    """
    Umfassende Evaluation der Backtesting-Ergebnisse.
    results_df: DataFrame mit kumulierten Werten (Equity Curves)
    trades_df: DataFrame mit den binären Signalen (um Handelsfrequenz zu messen)
    """
    stats = []
    
    for col in results_df.columns:
        equity_curve = results_df[col]
        daily_returns = equity_curve.pct_change().dropna()
        
        # 1. Total Return & CAGR (Annualisierte Rendite)
        total_return = (equity_curve.iloc[-1] / equity_curve.iloc[0]) - 1
        days = (equity_curve.index[-1] - equity_curve.index[0]).days
        cagr = (equity_curve.iloc[-1] / equity_curve.iloc[0])**(365.25/days) - 1
        
        # 2. Volatilität (annualisiert)
        vol = daily_returns.std() * np.sqrt(252)
        
        # 3. Sharpe Ratio (Annahme: Risk-Free Rate = 0, da Cash bereits in Strategie steckt)
        sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(252) if daily_returns.std() != 0 else 0
        
        # 4. Maximum Drawdown
        peak = equity_curve.expanding(min_periods=1).max()
        drawdown = (equity_curve / peak) - 1
        mdd = drawdown.min()
        
        # 5. Sortino Ratio (Fokus auf Downside-Risiko)
        downside_returns = daily_returns[daily_returns < 0]
        downside_std = downside_returns.std() * np.sqrt(252)
        sortino = (daily_returns.mean() * 252) / downside_std if downside_std != 0 else np.nan
        
        # 6. Calmar Ratio (Verhältnis Rendite zu Max Drawdown)
        calmar = cagr / abs(mdd) if mdd != 0 else np.nan
        
        # 7. Anzahl der Trades (Regime-Wechsel)
        # Wir messen, wie oft das Modell von 0 auf 1 oder umgekehrt springt
        if col in trades_df.columns:
            # Signaländerungen zählen (Absolutwert der Differenz)
            switches = trades_df[col].diff().abs().sum()
        else:
            switches = 0 # Buy & Hold hat 0 Wechsel
            
        stats.append({
            'Strategie': col,
            'Total Return': f"{total_return:.2%}",
            'CAGR (p.a.)': f"{cagr:.2%}",
            'Volatilität': f"{vol:.2%}",
            'Max Drawdown': f"{mdd:.2%}",
            'Sharpe Ratio': round(sharpe, 2),
            'Sortino Ratio': round(sortino, 2),
            'Calmar Ratio': round(calmar, 2),
            'Regime-Wechsel': int(switches)
        })
    
    return pd.DataFrame(stats).set_index('Strategie')

# Übergeben der Equity Curves und der Trade-Stratistik test_df mit den Signalen
# Mappen der Spaltennamen, damit die Funktion die Signale findet
signals_to_count = pd.DataFrame(index=test_df.index)
signals_to_count['MS_Univariate'] = test_df['MS_Univariate_Signal']
signals_to_count['MS_Exogenous'] = test_df['MS_Exo_Signal']
signals_to_count['LSTM_Regime'] = test_df['LSTM_Signal']
signals_to_count['HMM_Based'] = test_df['HMM_Regime']

evaluation_table = evaluate_strategies(backtesting_results, signals_to_count)

# Anzeige der Tabelle
print("\n--- UMFASSENDE EVALUATION ---")
display(evaluation_table)

# Die Tabelle in eine Markdown-Datei schreiben
evaluation_table.to_markdown('../assets/evaluation_table.md', index=True)
print("Evaluationstabelle erfolgreich unter ../assets/evaluation_table.md persistiert.")

# --- Wir erhalten in diesem Schritt neben df und test_df sowie backtesting_results_df trades_df mit binären Handelssignalen ---


--- UMFASSENDE EVALUATION ---


,Total Return,CAGR (p.a.),Volatilität,Max Drawdown,Sharpe Ratio,Sortino Ratio,Calmar Ratio,Regime-Wechsel
Strategie,,,,,,,,
Buy_Hold,92.90%,9.80%,12.62%,-27.10%,0.81,1.04,0.36,0
MS_Univariate,165.77%,14.93%,6.33%,-5.80%,2.24,2.88,2.57,42
MS_Exogenous,162.28%,14.71%,6.42%,-5.44%,2.18,2.79,2.70,38
LSTM_Regime,69.01%,7.76%,8.02%,-14.93%,0.97,1.12,0.52,70
HMM_Based,78.87%,8.63%,4.96%,-6.53%,1.70,1.56,1.32,29


Evaluationstabelle erfolgreich unter ../assets/evaluation_table.md persistiert.
